In [14]:
import pandas as pd
import os
import glob
from pathlib import Path
import statsmodels.api as sm
import numpy as np
import datetime

In [15]:
df = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\funds info\FUND_MainInfo.csv")

In [16]:
df["InceptionDate"] = pd.to_datetime(df["InceptionDate"], errors="coerce")
#df = df[df["InceptionDate"] <= datetime.datetime(2014,7,1)].copy()
#print(len(df))
df = df[(df["IsQDII"] == 2)]
#print(len(df))
df = df[(df["IsIndexFund"] == 2)]
#print(len(df))
df = df[(df["FundTypeID"] == 'S0501' )]
#print(len(df))
df = df[(df["CategoryID"] == 'S0604')| (df["CategoryID"] == 'S0601')]
#df = df[(df["CategoryID"] == 'S0604')]
#print(len(df))

In [17]:
folder = Path(r"D:\studystudy\Window Dressing\raw data\funds position")
files = sorted(folder.glob("Fund_Portfolio_Stock*.csv"))
df_all = pd.concat((pd.read_csv(f , low_memory=False) for f in files), ignore_index=True)

In [18]:
df_all = df_all[df_all["Proportion"].notna() & (df_all["Proportion"] != 0)].copy()
keep_codes = set(df['MasterFundCode'].dropna().unique())
df_all.drop(df_all[~df_all['MasterFundCode'].isin(keep_codes)].index, inplace=True)
df_all = df_all[df_all["ReportTypeID"].isin([5, 6])].reset_index(drop=True)
df_all.rename(columns={'Startdate': 'StartDate'}, inplace=True)

In [19]:
df_all["EndDate"] = pd.to_datetime(df_all["EndDate"], errors="coerce")
df_all["Year"] = df_all["EndDate"].dt.year
df_all["Half"] = df_all["ReportTypeID"].map({5: 1, 6: 2}).astype("Int64")
group_cols = ["MasterFundCode", "Year", "Half"]
stock_total = df_all.groupby(group_cols)["Proportion"].transform("sum")

df_all["Proportion_StockOnly"] = np.where(
    stock_total > 0,
    df_all["Proportion"] / stock_total * 100,
    np.nan)

In [20]:
df_stk = pd.read_csv(
    r"D:\studystudy\Window Dressing\merged data\merged_half_year_stk_ret.csv",
    usecols=["Symbol", "Year", "Half", "HalfReturn", "WinnerScore"],
    dtype={"Symbol": "string"}  # 确保不会被当成数字把前导0吞掉
)

df_stk["Symbol"] = df_stk["Symbol"].str.strip().str.zfill(6)

In [21]:
# 合并
before_rows = len(df_all)
df_all = df_all.merge(
    df_stk[["Symbol", "Year", "Half", "HalfReturn", "WinnerScore"]],
    on=["Symbol", "Year", "Half"],
    how="left",
    validate="many_to_one"   # df_all 可以多行同 key，df_stk 必须一行同 key
)
# 检验

In [22]:
'''print("Row count unchanged:", len(df_all) == before_rows)
print("Missing HalfReturn:", df_all["HalfReturn"].isna().sum())
print("Missing WinnerScore:", df_all["WinnerScore"].isna().sum())
print("Match rate (HalfReturn):", 1 - df_all["HalfReturn"].isna().mean())'''

'print("Row count unchanged:", len(df_all) == before_rows)\nprint("Missing HalfReturn:", df_all["HalfReturn"].isna().sum())\nprint("Missing WinnerScore:", df_all["WinnerScore"].isna().sum())\nprint("Match rate (HalfReturn):", 1 - df_all["HalfReturn"].isna().mean())'

In [23]:
'''# 1) 把 missing 的行调出来（HalfReturn 为空即视为没匹配到）
df_missing = df_all.loc[df_all["HalfReturn"].isna(), ["MasterFundCode", "Symbol", "Year", "Half"]].copy()
# 2) 统计 missing 的 key 数量（原始行 vs 唯一key）
print("Missing rows:", len(df_missing))
print("Missing unique keys:", df_missing.drop_duplicates(["Symbol", "Year", "Half"]).shape[0])
# 3) 核对：这些 missing key 是否真的不在 df_stk（应该全部为 True）
missing_keys = df_missing.drop_duplicates(["Symbol", "Year", "Half"])
check = missing_keys.merge(
    df_stk[["Symbol", "Year", "Half"]],
    on=["Symbol", "Year", "Half"],
    how="left",
    indicator=True
)
print(check["_merge"].value_counts())
# 如果看到全部是 'left_only'，说明 df_stk 里确实没有这些 (Symbol,Year,Half)
miss = df_all[df_all["HalfReturn"].isna()]
print(miss.groupby(["Year", "Half"]).size().sort_values(ascending=False).head(30))
print(miss["Symbol"].value_counts().head(20))'''

'# 1) 把 missing 的行调出来（HalfReturn 为空即视为没匹配到）\ndf_missing = df_all.loc[df_all["HalfReturn"].isna(), ["MasterFundCode", "Symbol", "Year", "Half"]].copy()\n# 2) 统计 missing 的 key 数量（原始行 vs 唯一key）\nprint("Missing rows:", len(df_missing))\nprint("Missing unique keys:", df_missing.drop_duplicates(["Symbol", "Year", "Half"]).shape[0])\n# 3) 核对：这些 missing key 是否真的不在 df_stk（应该全部为 True）\nmissing_keys = df_missing.drop_duplicates(["Symbol", "Year", "Half"])\ncheck = missing_keys.merge(\n    df_stk[["Symbol", "Year", "Half"]],\n    on=["Symbol", "Year", "Half"],\n    how="left",\n    indicator=True\n)\nprint(check["_merge"].value_counts())\n# 如果看到全部是 \'left_only\'，说明 df_stk 里确实没有这些 (Symbol,Year,Half)\nmiss = df_all[df_all["HalfReturn"].isna()]\nprint(miss.groupby(["Year", "Half"]).size().sort_values(ascending=False).head(30))\nprint(miss["Symbol"].value_counts().head(20))'

In [24]:
# 1) 直接删除 HalfReturn / WinnerScore 缺失，以及 Proportion<=0 或缺失的行
df_use = df_all.dropna(subset=["HalfReturn", "WinnerScore", "Proportion"]).copy()
df_use = df_use[df_use["Proportion"] > 0].copy()
# 2) 重新归一化权重（股票内部占比）
total = df_use.groupby(group_cols)["Proportion"].transform("sum")
df_use = df_use[total > 0].copy()  # 双保险：避免极端情况下分母为0
df_use["Proportion_StockOnly"] = df_use["Proportion"] / total * 100

In [25]:
group_cols = ["MasterFundCode", "Year", "Half"]
def wavg(g, value_col):
    w = g["Proportion_StockOnly"]
    v = g[value_col]
    s = w.sum()
    return np.nan if s == 0 else (w * v).sum() / s
df_fund_period = (df_use
    .groupby(group_cols, as_index=False)
    .apply(lambda g: pd.Series({
        "WeightedHalfReturn": wavg(g, "HalfReturn"),
        "WeightedWinnerScore": wavg(g, "WinnerScore"),
        "WeightSum_used_for_return": g["Proportion_StockOnly"].sum(),
        "N_stocks": len(g)})))
df_fund_period.drop(columns = ['WeightSum_used_for_return','N_stocks'], inplace = True)

C:\Users\MLTZ\AppData\Local\Temp\ipykernel_22968\1113802654.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [26]:
df_ic = pd.read_csv(r'D:\studystudy\Window Dressing\raw data\funds_ag\Fund_FIN_Income.csv')
df_ba = pd.read_csv(r'D:\studystudy\Window Dressing\raw data\funds_ag\FUND_FIN_Balance.csv')

In [27]:
df_ic = df_ic[df_ic['DATA_TYPE_ID']==3]
df_ic["EndDate"] = pd.to_datetime(df_ic["EndDate"], errors="coerce")
df_ba["EndDate"] = pd.to_datetime(df_ba["EndDate"], errors="coerce")
df_ic["Year"] = df_ic["EndDate"].dt.year
df_ba["Year"] = df_ba["EndDate"].dt.year
df_ic["Half"] = df_ic["ReportTypeID"].map({5: 1, 6: 2}).astype("Int64")
df_ba["Half"] = df_ba["ReportTypeID"].map({5: 1, 6: 2}).astype("Int64")
df_ic.dropna(inplace = True)
df_ba.dropna(inplace = True)
df_ic = (
    df_ic.groupby(["MasterFundCode", "Year", "Half"], as_index=False)[
        ["StockInvestmentIncome", "StockDividend"]
    ].sum())
cols = ["StockInvestmentIncome", "StockDividend"]
# 取同基金同年的上半年累计（Half=1）
h1 = (df_ic[df_ic["Half"] == 1][["MasterFundCode", "Year"] + cols]
      .rename(columns={c: f"{c}_H1" for c in cols}))
# 并回原表
df_ic = df_ic.merge(h1, on=["MasterFundCode", "Year"], how="left")
# 计算当期半年收入：H1 原值；H2 用差分
for c in cols:
    df_ic[c] = np.where(
        df_ic["Half"] == 2,
        df_ic[c] - df_ic[f"{c}_H1"],
        df_ic[c])
df_ic.drop(columns=[f"{c}_H1" for c in cols], inplace=True)
df_ba = (
    df_ba.sort_values("EndDate")
         .groupby(["MasterFundCode", "Year", "Half"], as_index=False)
         .tail(1)[["MasterFundCode", "Year", "Half", "StockValue"]]
         .reset_index(drop=True))
df_icba = df_ic.merge(
    df_ba[["MasterFundCode", "Year", "Half", "StockValue"]],
    on=["MasterFundCode", "Year", "Half"],
    how="inner")
df_icba = df_icba[['MasterFundCode','Year','Half','StockInvestmentIncome','StockDividend','StockValue']]
df_icba = df_icba[df_icba['StockValue'] > 0].copy()
df_icba['AG'] = (df_icba['StockInvestmentIncome']+df_icba['StockDividend'])/df_icba['StockValue']

In [28]:
df_icba.drop(df_icba[~df_icba['MasterFundCode'].isin(keep_codes)].index, inplace=True)

In [29]:
#print(df_fund_period['WeightedHalfReturn'].describe())
#print(df_icba['AG'].describe())


In [30]:
df_bhrg = df_fund_period.merge(
    df_icba[["MasterFundCode", "Year", "Half", "AG"]],
    on=["MasterFundCode", "Year", "Half"],
    how="inner")
df_bhrg['bhrg'] = df_bhrg['WeightedHalfReturn'] - df_bhrg['AG']

In [31]:
group_cols = ["Year", "Half"]
# 组内基金数量
n = df_bhrg.groupby(group_cols)["AG"].transform("size")
# 组内倒序排名（数值越大名次越靠前）
df_bhrg["performance_rank_raw"] = df_bhrg.groupby(group_cols)["AG"].rank(ascending=False, method="average")
df_bhrg["score_rank_raw"] = df_bhrg.groupby(group_cols)["WeightedWinnerScore"].rank(ascending=False, method="average")
# 名次 -> [0,1] 分数：第一名=0，最后一名=1（均匀）
den = n - 1
df_bhrg["performance_rank"] = np.where(den > 0, (df_bhrg["performance_rank_raw"] - 1) / den, 0.0)
df_bhrg["score_rank"]       = np.where(den > 0, (df_bhrg["score_rank_raw"]       - 1) / den, 0.0)
df_bhrg["RankGap"] = df_bhrg["performance_rank"] - df_bhrg["score_rank"]
# 如果你不需要保留 raw 名次列，可以删掉
df_bhrg.drop(columns=["performance_rank_raw", "score_rank_raw"], inplace=True)

In [32]:
df_index = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\index\CAPMR_Idxdalyr.csv")
df_index = df_index[df_index['Indexcd'] == 1]
df_index = df_index.copy()
df_index["Trddt"] = pd.to_datetime(df_index["Trddt"])
df_index["Year"] = df_index["Trddt"].dt.year
df_index["Half"] = np.where(df_index["Trddt"].dt.month <= 6, 1, 2)
df_half_return = (
    df_index
    .groupby(["Year", "Half"], as_index=False)
    .agg(
        HalfReturn=("Retindex", lambda x: (1 + x).prod() - 1),
        #N_days=("Retindex", "size")   # 可选：该半年交易日数，用于审计
    )
)

In [33]:
df_temp = df_bhrg[['MasterFundCode','Year','Half','bhrg']].copy()
df_reg_temp = df_temp.merge(df_half_return , on = ['Year','Half'] , how = 'left' )
# 1) 取出需要回归的样本（去掉缺失）
tmp = df_reg_temp[["bhrg", "HalfReturn"]].dropna().copy()

# 2) OLS: bhrg = alpha + beta * HalfReturn + error
X = sm.add_constant(tmp["HalfReturn"])
y = tmp["bhrg"]

model = sm.OLS(y, X).fit()

# 3) 残差回填到原df（未参与回归的行保持为 NaN）
df_reg_temp["resBHRG"] = np.nan
df_reg_temp.loc[tmp.index, "resBHRG"] = model.resid

In [34]:
df_bhrg = df_bhrg.merge(df_reg_temp[['MasterFundCode','Year','Half','resBHRG']], on = ['MasterFundCode','Year','Half'],how = 'left')
df_final = df_bhrg.drop(columns = ['WeightedHalfReturn','WeightedWinnerScore','AG','performance_rank','score_rank'])
df_final.rename(columns={'bhrg': 'BHRG'}, inplace=True)

In [35]:
df_dup_rows = df_final[
    df_final.duplicated(subset=["MasterFundCode", "Year", "Half"], keep=False)
].sort_values(["MasterFundCode", "Year", "Half"])
print(df_dup_rows)


Empty DataFrame
Columns: [MasterFundCode, Year, Half, BHRG, RankGap, resBHRG]
Index: []


In [36]:
df_final.to_csv(r'D:\studystudy\Window Dressing\merged data\merged_DependentVariable.csv', index=False, sep=',', encoding='utf-8-sig')

# ManagerSkill

In [37]:
df_fr1 = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\Fund_NAV_Month.csv")
df_fr2 = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\Fund_NAV_Month1.csv")
df_fr = pd.concat([df_fr1,df_fr2])
df_temp = pd.read_csv(r"D:\studystudy\Window Dressing\merged data\merged_DependentVariable.csv")
df_s2m = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\FUND_FundCodeInfo.csv")
# 1) Symbol -> MasterFundCode 映射（保留 df_fr 的所有列，并新增 MasterFundCode）
map_s2m = df_s2m[['Symbol', 'MasterFundCode']].drop_duplicates(subset=['Symbol'])
df_fr_m = df_fr.merge(map_s2m, on='Symbol', how='left')
# 2) 用 df_temp 的 MasterFundCode 过滤（只保留能在因变量表里对上的主基金）
keep_master = set(df_temp['MasterFundCode'].dropna().unique())
df_fr_m = df_fr_m[df_fr_m['MasterFundCode'].isin(keep_master)].copy()

In [38]:
df_fr_m = df_fr_m.copy()
df_fr_m["TradingDate"] = pd.to_datetime(df_fr_m["TradingDate"], errors="coerce")

# 生成 Year / Half（按交易日期）
df_fr_m["Year"] = df_fr_m["TradingDate"].dt.year
df_fr_m["Half"] = np.where(df_fr_m["TradingDate"].dt.month <= 6, 1, 2)

# 月度收益率列（你现成的）
df_fr_m["Ret_m"] = pd.to_numeric(df_fr_m["ReturnAccumulativeNAV"], errors="coerce")

# 半年复利收益：prod(1+Ret_m) - 1
df_fund_half = (
    df_fr_m.dropna(subset=["MasterFundCode", "Year", "Half", "Ret_m"])
          .groupby(["MasterFundCode", "Year", "Half"], as_index=False)
          .agg(
              FundHalfReturn=("Ret_m", lambda x: (1 + x).prod() - 1),
              N_months=("Ret_m", "size")))

In [39]:
# 1) 取出上一期权重（t-1）并把期数变为可+1的序号
w = df_use[["MasterFundCode", "Symbol", "Year", "Half", "Proportion_StockOnly"]].copy()
w["period"] = w["Year"].astype(int) * 2 + w["Half"].astype(int)
w["period_next"] = w["period"] + 1
# 2) 把 period_next 还原成 (Year, Half)，这就是“用t-1权重去解释的t期”
w["Year_t"] = (w["period_next"] - 1) // 2
w["Half_t"] = (w["period_next"] - 1) % 2 + 1
# 3) 合并本期股票收益（t）
w = w.merge(
    df_stk[["Symbol", "Year", "Half", "HalfReturn"]],
    left_on=["Symbol", "Year_t", "Half_t"],
    right_on=["Symbol", "Year", "Half"],
    how="inner" )
# 4) 计算假设收益：按基金-期 (t) 汇总 Σ(w_{t-1} * r_t) / 100
w["contrib"] = w["Proportion_StockOnly"] * w["HalfReturn"] / 100.0

df_hypo = (
    w.groupby(["MasterFundCode", "Year_t", "Half_t"], as_index=False)["contrib"]
     .sum()
     .rename(columns={"Year_t": "Year", "Half_t": "Half", "contrib": "HypoReturn"})
)
# 5) 真实收益（按你的定义：与AG一样）
df_actual = df_icba[["MasterFundCode", "Year", "Half", "AG"]].copy()
df_actual = df_actual.rename(columns={"AG": "ActualReturn"})

# 6) 合并并计算 ManagerSkill_{i,t}
df_skill = df_actual.merge(df_hypo, on=["MasterFundCode", "Year", "Half"], how="inner")
df_skill["ManagerSkill"] = df_skill["ActualReturn"] - df_skill["HypoReturn"]

# 7) 生成 ManagerSkill_{i,t-1}（基金内按期排序后 shift(1)）
df_skill["period"] = df_skill["Year"].astype(int) * 2 + df_skill["Half"].astype(int)
df_skill = df_skill.sort_values(["MasterFundCode", "period"])

df_skill["ManagerSkill_t_1"] = df_skill.groupby("MasterFundCode")["ManagerSkill"].shift(1)

# 如果你只需要 i,t-1 指标用于回归，可以保留这些列
df_skill = df_skill[["MasterFundCode", "Year", "Half", "ActualReturn", "HypoReturn",
                     "ManagerSkill", "ManagerSkill_t_1"]]

In [40]:
df_skill.to_csv(r'D:\studystudy\Window Dressing\raw data\independent variable\others\ManagerSkill.csv', index=False, sep=',', encoding='utf-8-sig')

In [41]:
df_dup_rows = df_skill[
    df_skill.duplicated(subset=["MasterFundCode", "Year", "Half"], keep=False)
].sort_values(["MasterFundCode", "Year", "Half"])

print(df_dup_rows)


Empty DataFrame
Columns: [MasterFundCode, Year, Half, ActualReturn, HypoReturn, ManagerSkill, ManagerSkill_t_1]
Index: []
